In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from dti.utils import load_embedding

torch.set_grad_enabled(False)

/local2/kunkim/dti/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    dtype=torch.bfloat16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.17it/s]


In [3]:
state_dict = "../output/dti-sana1.5_1.6b-0920-debug/dog/learned_embeds.safetensors"

load_embedding(
    tokenizer=tokenizer,
    text_encoder=model,
    state_dict=state_dict,
)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


['<dog>']

In [12]:
input_text = "<dog> What is the Machine Learning?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")
print(input_ids)

outputs = model.generate(
    **input_ids, max_new_tokens=100, do_sample=True, top_p=0.9, temperature=0.7
)
print(outputs)
print(tokenizer.decode(outputs[0]))

{'input_ids': tensor([[     2, 256000,   2439,    603,    573,  13403,  14715, 235336]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
tensor([[     2, 256000,   2439,    603,    573,  13403,  14715, 235336, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
         256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000, 256000,
     

In [ ]:
from safetensors.torch import load_file


path = "../output/dti-sana1.5_1.6b-0921-newimpl/backpack_dog/learned_embeds-steps-1000.safetensors"
# path = "../output/dti-sana1.5_1.6b-0921-debug/cat/learned_embeds-steps-1000.safetensors"
state_dict = load_file(path)
print(state_dict.keys())
for k, v in state_dict.items():
    print(k, v.norm())

dict_keys(['<backpack_dog>'])
<backpack_dog> tensor(6.8943)


In [2]:
from transformers import Gemma2Model, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-2-2b-it",
)
model = Gemma2Model.from_pretrained(
    "google/gemma-2-2b-it",
)

/local2/kunkim/dti/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.88it/s]


In [8]:
embeddings = model.get_input_embeddings()

norms = embeddings.weight.norm(dim=1)
print(norms.min())
print(norms.max())
print(norms.mean())

# Decode max-norm and min-norm embeddings
max_norm_id = norms.argmax().item()
min_norm_id = norms.argmin().item()
print("max-norm token:", max_norm_id, tokenizer.decode([max_norm_id]))

# Compute the norm of "dog" token
target = "vase"
target_id = tokenizer.convert_tokens_to_ids(target)
print(
    f"{target} token:",
    target_id,
    tokenizer.decode([target_id]),
    embeddings.weight[target_id].norm().item(),
)

tensor(1.0645, grad_fn=<MinBackward1>)
tensor(4.8634, grad_fn=<MaxBackward1>)
tensor(1.7720, grad_fn=<MeanBackward0>)
max-norm token: 234323  myſelf
vase token: 123002 vase 1.8508399724960327
